This note is taken from *Human-in-the-loop Machine learning*. Breifly on active learning and quality control and all, which aims at balancing cost benefit.

0. Unlabeled Data (unlabeled_data.csv)
        v
1. Hold out data for evaluation
        v
2. Active Learning sampling
        v
3. Annotate
        v
4. Added to training_data/related.csv or not_related.csv
        v
5. Model training save to (model/$(date +"%Y%m%d_%H%M%S"_F-score_AUC_training-size.params)
        v
Repeat

In [5]:
# Project path setup
%cd /content/drive/MyDrive/pytorch_active_learning
models_path = '/content/drive/MyDrive/pytorch_active_learning/models/'

/content/drive/MyDrive/pytorch_active_learning


In [6]:
# import Python libraries
import os
import glob
import pandas as pd
from datetime import datetime
from IPython.display import display, Markdown


In [7]:
# parser function to retrieve model training time, f-score, AUC, and sample size from trained model name
def get_models_info(models_path=models_path):
    """Parse all model files and return DataFrame with info"""

    model_files = glob.glob(f"{models_path}*.params")
    models_data = []

    for model_path in sorted(model_files):
        filename = os.path.basename(model_path)
        # Remove .params and split by underscore
        parts = filename.replace('.params', '').split('_')

        if len(parts) >= 5:
            try:
                date_str = f"{parts[0]}_{parts[1]}"
                timestamp = datetime.strptime(date_str, '%Y%m%d_%H%M%S')

                model_info = {
                    'filename': filename,
                    'full_path': model_path,
                    'date': timestamp,
                    'fscore': float(parts[2]),
                    'auc': float(parts[3]),
                    'training_size': int(parts[4]),
                    'order': len(models_data) + 1
                }
                models_data.append(model_info)
            except (ValueError, IndexError) as e:
                print(f"Warning: Could not parse {filename}: {e}")

    return pd.DataFrame(models_data)


# Get model by index
def get_model_by_index(index, models_path=models_path):
    """Get model info by order (0 = oldest, -1 = newest)"""
    models_df = get_models_info(models_path)

    if models_df.empty:
        return None

    if index == -1:
        return models_df.iloc[index]  # Latest
    elif index >= 0:
        return models_df.iloc[index] if index < len(models_df) else None
    else:
        return models_df.iloc[index] if abs(index) <= len(models_df) else None




In [15]:
# format text for notebook output
def format_model_result(model_info):
    """Create formatted text for Jupyter markdown"""

    if model_info is None:
        return "No model found"

    iteration_description = model_info['order']

    text = f"""
### Iteration: {iteration_description}

**Model File:** `{model_info['filename']}`

**Training Results:**
- **F-Score:** `{model_info['fscore']:.4f}`
- **AUC:** `{model_info['auc']:.4f}`
- **Training Size:** `{model_info['training_size']}` samples
- **Date/Time:** `{model_info['date'].strftime('%Y-%m-%d %H:%M:%S')}`

**Interpretation:**
- F-Score = {model_info['fscore']:.4f} hints {interpret_fscore(model_info['fscore'])}
- AUC = {model_info['auc']:.4f} hints {interpret_auc(model_info['auc'])}

---
"""
    return text

def interpret_fscore(fscore):
    if fscore == 0:
        return "Model failed completely"
    elif fscore < 0.5:
        return "Model performs poorly"
    elif fscore < 0.7:
        return "Model shows some ability but needs improvement"
    elif fscore < 0.8:
        return "Model is reasonably effective"
    elif fscore < 0.9:
        return "Model performs well"
    else:
        return "Model performs excellently"

def interpret_auc(auc):
    if auc < 0.5:
        return "Model is worse than random (something is wrong)"
    elif auc < 0.6:
        return "Model barely above random guessing"
    elif auc < 0.7:
        return "Model shows weak discrimination ability"
    elif auc < 0.8:
        return "Model has acceptable discrimination"
    elif auc < 0.9:
        return "Model has good discrimination"
    else:
        return "Model has excellent discrimination"

In [17]:
# Get models
models_df = get_models_info()
markdown_text = format_model_result(models_df.iloc[0])

# Display as formatted markdown
display(Markdown(markdown_text))


### Iteration: 1

**Model File:** `20260522_142811_0.0_0.668_400.params`

**Training Results:**
- **F-Score:** `0.0000`
- **AUC:** `0.6680`
- **Training Size:** `400` samples
- **Date/Time:** `2026-05-22 14:28:11`

**Interpretation:**
- F-Score = 0.0000 hints Model failed completely
- AUC = 0.6680 hints Model shows weak discrimination ability

---


In [19]:
!wc -l ./training_data/*

  354 ./training_data/not_related.csv
   58 ./training_data/related.csv
  412 total


In [ ]:
import math
import numpy as np
import torch as torch

In [ ]:
zscore = [1,4,2,3]

In [ ]:
def softmax(arr,base=math.e):
    ex = [math.pow(base,x) for x in arr]
    denominator = sum(ex)
    return [x / denominator for x in ex]

confidence = softmax(zscore)
prob = torch.tensor(confidence)
# [round(x,10) for x in softmax(zscore)] == [round(x,10) for x in softmax([-2,1,-1,0])]


# Uncertainty sampling
1. least confidence


In [ ]:
most_conf = torch.max(prob)
num_labels = prob.numel()   # length of array
numerator = (num_labels * (1 - most_conf))  # 1 for most uncertainty
denomitor = (num_labels - 1)

least_conf = numerator / denomitor
least_conf

tensor(0.4748)

2. margin_conf

In [ ]:
prob, _ = torch.sort(prob,descending=True)
difference = (prob.data[0] - prob.data[1])
margin_conf = 1 - difference
margin_conf

tensor(0.5930)

3. Ratio of confidence

In [ ]:
ratio_conf = prob.data[1] / prob.data[0]
ratio_conf

tensor(0.3679)

4. entropy

In [ ]:
prbslogs = prob * torch.log2(prob)
numerator = 0 - torch.sum(prbslogs)
denominator = torch.log2(torch.tensor(prob.numel()))
entropy = numerator / denominator
entropy

tensor(0.6835)

In [ ]:
!python active_learning.py

/content/drive/MyDrive/pytorch_active_learning/active_learning.py:402: SyntaxWarning: invalid escape sequence '\.'
  timestamp = re.sub('\.[0-9]*','_',str(datetime.datetime.now())).replace(" ", "_").replace("-", "").replace(":","")
Traceback (most recent call last):
  File "/content/drive/MyDrive/pytorch_active_learning/active_learning.py", line 30, in <module>
    from diversity_sampling import DiversitySampling
  File "/content/drive/MyDrive/pytorch_active_learning/diversity_sampling.py", line 35, in <module>
    from uncertainty_sampling import UncertaintySampling
  File "/content/drive/MyDrive/pytorch_active_learning/uncertainty_sampling.py", line 35
    checkpoint_mgr = ActiveLearningCheckpoint(
IndentationError: unexpected indent
